**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 16: 합성 데이터 생성 & Distillation

**Part 3: 파인튜닝 기초** | GPU: No (API 기반)

---

### 📋 학습 목표
- 🎯 합성 데이터(Synthetic Data) 생성의 필요성과 방법론을 이해한다
- 🎯 Self-Instruct 방법론의 원리를 파악한다
- 🎯 GPT-4를 활용하여 학습 데이터를 자동 생성한다
- 🎯 Knowledge Distillation의 개념과 활용법을 학습한다
- 🎯 생성 데이터의 품질 검증 및 비용 최적화 방법을 익힌다

### ⏱️ 예상 소요 시간: 90분

### 💰 예상 API 비용: ~$0.5~1.0 (GPT-4o-mini 기준)

In [ ]:
# 환경 설정
import json
import os
import time
import random
from typing import List, Dict

print("✅ Session 16: 합성 데이터 생성 & Distillation")
print("📌 이 노트북은 OpenAI API를 사용합니다 (GPU 불필요)")
print()

# 데이터 디렉토리
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "samples")
os.makedirs(DATA_DIR, exist_ok=True)

# OpenAI API 설정
try:
    from openai import OpenAI
    client = OpenAI()  # OPENAI_API_KEY 환경변수 사용
    print("✅ OpenAI 클라이언트 초기화 성공")
except ImportError:
    print("⚠️ openai 라이브러리가 설치되어 있지 않습니다.")
    print("   설치: pip install openai")
    client = None
except Exception as e:
    print(f"⚠️ OpenAI 클라이언트 초기화 실패: {e}")
    print("   OPENAI_API_KEY 환경변수를 설정하세요.")
    client = None

---
## 🎯 1. 합성 데이터와 Distillation 개요

### 합성 데이터(Synthetic Data)란?

실제 데이터가 아닌, **AI 모델이 생성한 인공 데이터**를 의미합니다.

```
🧠 Teacher Model (GPT-4)
    ↓ 데이터 생성
📚 합성 학습 데이터 (수천~수만개)
    ↓ 학습
🤖 Student Model (Qwen2.5-1.5B)
```

### Knowledge Distillation이란?

큰 모델(Teacher)의 **지식을 작은 모델(Student)**에게 전달하는 기법

| 항목 | Teacher | Student |
|------|---------|----------|
| 모델 | GPT-4, Claude | Qwen2.5-1.5B |
| 파라미터 | 수천억 | 15억 |
| 비용 | API 비용 | 로컬 무료 |
| 속도 | 느림 | 빠름 |
| 프라이버시 | 외부 서버 | 로컬 환경 |

---
## 1️⃣ 2. 합성 데이터 생성이 필요한 이유

### 현실적인 문제들

| 문제 | 설명 | 합성 데이터 해결 |
|------|------|----------------|
| 📌 **데이터 부족** | 도메인 특화 데이터가 적음 | LLM으로 대량 생성 |
| 📌 **비용 문제** | 전문가 라벨링 비용 높음 | API 비용으로 대체 |
| 📌 **프라이버시** | 실제 데이터 활용 제한 | 합성 데이터로 우회 |
| 📌 **다양성 부족** | 편향된 데이터 | 다양한 시나리오 생성 |
| 📌 **시간 문제** | 데이터 수집에 수개월 | 수시간~수일 |

### 합성 데이터의 성공 사례

- 🏆 **Alpaca (Stanford)**: GPT-3.5로 52K 데이터 생성 → 7B 모델 파인튜닝
- 🏆 **Vicuna**: ShareGPT 대화 데이터 → 13B 모델이 GPT-4의 90% 성능
- 🏆 **Orca**: GPT-4의 추론 과정 학습 → 작은 모델의 추론 능력 향상
- 🏆 **phi-1.5 (Microsoft)**: 합성 교과서 데이터로 1.3B 모델이 5~25B 모델 성능

---
## 2️⃣ 3. Self-Instruct 방법론

### Self-Instruct 프로세스

2022년 Wang et al.이 제안한 방법으로, LLM이 스스로 학습 데이터를 생성하는 부트스트래핑 기법입니다.

```
📝 Seed Instructions (초기 시드 데이터 ~175개)
    ↓
🤖 LLM이 새로운 Instruction 생성
    ↓
🔍 품질 필터링 (중복 제거, 유사도 체크)
    ↓
🤖 LLM이 각 Instruction에 대한 Output 생성
    ↓
📚 새로운 학습 데이터 (52K+)
    ↓
🔄 반복 (새 데이터를 시드에 추가)
```

### 핵심 원리

1. **다양성 확보**: 시드 데이터의 카테고리를 다양하게
2. **품질 필터링**: 생성된 데이터 중 고품질만 선별
3. **반복 확장**: 좋은 데이터를 시드에 추가하여 재귀적 확장

In [ ]:
# Seed 데이터 정의
print("📝 Seed 데이터 정의 (다양한 카테고리)")
print("=" * 60)

seed_data = [
    {
        "instruction": "다음 문장을 영어로 번역하세요.",
        "input": "오늘 날씨가 매우 좋습니다.",
        "output": "The weather is very nice today.",
        "category": "번역"
    },
    {
        "instruction": "다음 텍스트의 감성을 분석하세요.",
        "input": "이 영화는 정말 재미있었어요! 강력 추천합니다.",
        "output": "긍정적인 감성입니다. '재미있었어요', '강력 추천' 등의 표현이 사용되었습니다.",
        "category": "감성분석"
    },
    {
        "instruction": "파이썬에서 리스트를 정렬하는 방법을 설명하세요.",
        "input": "",
        "output": "파이썬에서 리스트 정렬은 두 가지 방법이 있습니다:\n1. sort() 메서드: 원본 리스트를 직접 정렬 (my_list.sort())\n2. sorted() 함수: 새로운 정렬된 리스트 반환 (sorted(my_list))\n역순 정렬은 reverse=True 파라미터를 사용합니다.",
        "category": "코딩"
    },
    {
        "instruction": "다음 내용을 3줄로 요약하세요.",
        "input": "인공지능은 1950년대부터 연구되어 왔으며, 최근 딥러닝의 발전으로 급격한 성능 향상을 이루었습니다. 특히 트랜스포머 아키텍처의 등장은 자연어처리 분야를 혁신했습니다.",
        "output": "1. AI는 1950년대부터 연구된 분야입니다.\n2. 딥러닝의 발전이 AI 성능을 급격히 향상시켰습니다.\n3. 트랜스포머는 NLP 분야를 혁신했습니다.",
        "category": "요약"
    },
    {
        "instruction": "비즈니스 이메일을 작성하세요.",
        "input": "프로젝트 미팅 일정 확인 요청",
        "output": "제목: 프로젝트 미팅 일정 확인\n\n안녕하세요,\n\n다음 주 예정된 프로젝트 진행 미팅의 일시와 장소를 확인하고자 합니다.\n참석 가능한 일정을 회신해 주시면 감사하겠습니다.\n\n감사합니다.",
        "category": "비즈니스"
    },
    {
        "instruction": "머신러닝에서 과적합이란 무엇인지 설명하세요.",
        "input": "",
        "output": "과적합(Overfitting)은 모델이 학습 데이터에 지나치게 맞춰져 새로운 데이터에 대한 성능이 떨어지는 현상입니다. 해결 방법으로는 더 많은 데이터 수집, 정규화 적용, 드롭아웃 사용, 조기 종료 등이 있습니다.",
        "category": "설명"
    },
    {
        "instruction": "주어진 키워드로 짧은 시를 작성하세요.",
        "input": "가을, 낙엽, 바람",
        "output": "가을 바람이 불어와\n나뭇잎을 흔들고\n붉은 낙엽이 하나둘\n춤추며 떨어진다\n계절이 바뀌어가는\n자연의 노래",
        "category": "창작"
    },
    {
        "instruction": "다음 문장의 문법 오류를 찾아 수정하세요.",
        "input": "나는 어제 학교에 갔다 왔습니다.",
        "output": "수정: '나는 어제 학교에 다녀왔습니다.' 또는 '저는 어제 학교에 갔다 왔습니다.'\n\n설명: '나는'과 '습니다'의 존댓말 수준이 불일치합니다. '나는'(반말) + '왔습니다'(존댓말) → '저는' + '왔습니다' 또는 '나는' + '왔다'로 통일해야 합니다.",
        "category": "교정"
    },
    {
        "instruction": "다음 수학 문제를 풀어주세요.",
        "input": "한 가게에서 사과 3개에 1500원입니다. 10개를 사면 얼마인가요?",
        "output": "풀이:\n1. 사과 1개 가격: 1500원 ÷ 3 = 500원\n2. 사과 10개 가격: 500원 × 10 = 5,000원\n\n답: 5,000원",
        "category": "수학"
    },
    {
        "instruction": "다음 단어의 뜻을 설명하세요.",
        "input": "세렌디피티",
        "output": "세렌디피티(Serendipity)는 뜻밖의 행운, 우연한 발견을 의미하는 영어 단어입니다. 무언가를 찾다가 예상치 못한 더 좋은 것을 발견하는 상황을 가리킵니다. 예를 들어, 페니실린의 발견이 대표적인 세렌디피티입니다.",
        "category": "어휘"
    }
]

print(f"✅ Seed 데이터: {len(seed_data)}개")
print(f"\n📊 카테고리 분포:")
for item in seed_data:
    print(f"  🔹 [{item['category']}] {item['instruction'][:50]}")

---
## 3️⃣ 4. GPT-4를 활용한 데이터 생성 (OpenAI API)

In [ ]:
# 데이터 생성 프롬프트 설계
print("🔧 데이터 생성 프롬프트 설계")
print("=" * 60)

GENERATION_SYSTEM_PROMPT = """당신은 AI 학습 데이터 생성 전문가입니다.
주어진 시드 데이터를 참고하여, 새로운 instruction-output 쌍을 생성합니다.

규칙:
1. 시드 데이터와 유사하지만 다른 새로운 질문을 만드세요
2. 카테고리를 다양하게 포함하세요
3. 한국어로 작성하세요
4. output은 구체적이고 정확해야 합니다
5. 반드시 JSON 배열 형식으로 출력하세요"""

def create_generation_prompt(seed_examples, n_generate=5, category=None):
    """데이터 생성 프롬프트 생성"""
    # 시드 예시 선택 (랜덤 3개)
    if category:
        examples = [s for s in seed_examples if s['category'] == category]
        if not examples:
            examples = random.sample(seed_examples, min(3, len(seed_examples)))
    else:
        examples = random.sample(seed_examples, min(3, len(seed_examples)))
    
    example_text = json.dumps([
        {"instruction": e["instruction"], "input": e["input"], "output": e["output"]}
        for e in examples
    ], ensure_ascii=False, indent=2)
    
    category_hint = f"\n카테고리: {category}" if category else "\n다양한 카테고리를 포함하세요."
    
    prompt = f"""다음은 기존 학습 데이터 예시입니다:

{example_text}

위 예시를 참고하여, 새로운 {n_generate}개의 instruction-output 쌍을 생성하세요.
{category_hint}

중요:
- 기존 예시와 동일하지 않은 새로운 질문을 만드세요
- input이 필요한 경우와 필요하지 않은 경우를 모두 포함하세요
- output은 구체적이고 정확해야 합니다
- 반드시 JSON 배열 형식으로만 출력하세요 (다른 텍스트 없이)

JSON 형식:
[{{"instruction": "...", "input": "...", "output": "..."}}, ...]"""
    
    return prompt

# 프롬프트 예시 출력
sample_prompt = create_generation_prompt(seed_data, n_generate=3, category="코딩")
print("📋 생성 프롬프트 예시:")
print("-" * 60)
print(sample_prompt[:500] + "...")
print("\n💡 이 프롬프트를 GPT-4에 전송하면 새로운 학습 데이터가 생성됩니다.")

In [ ]:
# GPT-4를 활용한 데이터 생성 함수
print("🚀 GPT-4를 활용한 데이터 생성")
print("=" * 60)

def generate_data_batch(client, seed_examples, n_generate=5, 
                        category=None, model="gpt-4o-mini"):
    """GPT-4를 사용하여 데이터 배치 생성"""
    if client is None:
        print("⚠️ OpenAI 클라이언트가 없어 더미 데이터를 반환합니다.")
        return []
    
    prompt = create_generation_prompt(seed_examples, n_generate, category)
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": GENERATION_SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            temperature=0.8,  # 다양성을 위해 약간 높은 temperature
            max_tokens=2000,
        )
        
        content = response.choices[0].message.content.strip()
        
        # JSON 파싱 시도
        # 코드 블록 제거
        if content.startswith("```"):
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
        
        generated = json.loads(content)
        
        # 토큰 사용량 추적
        usage = response.usage
        return generated, {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens
        }
        
    except json.JSONDecodeError as e:
        print(f"  ⚠️ JSON 파싱 실패: {e}")
        return [], {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    except Exception as e:
        print(f"  ❌ API 호출 실패: {e}")
        return [], {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

print("✅ 데이터 생성 함수 정의 완료")
print("\n📌 함수 사용법:")
print('   data, usage = generate_data_batch(client, seed_data, n_generate=5, category="코딩")')

---
## 4️⃣ 5. 실습: Seed 기반 데이터 확장 (10개 → 100개)

In [ ]:
# 대규모 데이터 생성 파이프라인
print("🚀 Seed 기반 데이터 확장 파이프라인")
print("=" * 60)

categories = ["번역", "감성분석", "코딩", "요약", "비즈니스", 
              "설명", "창작", "교정", "수학", "어휘"]

all_generated = []
total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

if client is not None:
    print(f"📌 {len(categories)}개 카테고리 × 10개 = 총 100개 생성 목표")
    print(f"📌 모델: gpt-4o-mini")
    print()
    
    for i, category in enumerate(categories):
        print(f"  [{i+1}/{len(categories)}] 카테고리: {category}...", end=" ")
        
        generated, usage = generate_data_batch(
            client, seed_data, 
            n_generate=10, 
            category=category,
            model="gpt-4o-mini"
        )
        
        if generated:
            # 카테고리 태그 추가
            for item in generated:
                item["category"] = category
            all_generated.extend(generated)
            
            # 사용량 누적
            for key in total_usage:
                total_usage[key] += usage.get(key, 0)
            
            print(f"✅ {len(generated)}개 생성")
        else:
            print(f"❌ 생성 실패")
        
        time.sleep(1)  # Rate limiting
    
    print(f"\n{'='*60}")
    print(f"📊 생성 결과 요약:")
    print(f"  총 생성 데이터: {len(all_generated)}개")
    print(f"  총 토큰 사용: {total_usage['total_tokens']:,}")
    print(f"  입력 토큰: {total_usage['prompt_tokens']:,}")
    print(f"  출력 토큰: {total_usage['completion_tokens']:,}")
    
    # 비용 추정 (gpt-4o-mini 기준)
    input_cost = total_usage['prompt_tokens'] / 1_000_000 * 0.15
    output_cost = total_usage['completion_tokens'] / 1_000_000 * 0.60
    total_cost = input_cost + output_cost
    print(f"\n  💰 예상 비용: ${total_cost:.4f}")
    print(f"     입력: ${input_cost:.4f}")
    print(f"     출력: ${output_cost:.4f}")

else:
    print("⚠️ OpenAI API를 사용할 수 없어 더미 데이터를 사용합니다.")
    print("   실제 실습에서는 API 키를 설정한 후 실행하세요.")
    
    # 더미 데이터 (API 없이도 다음 단계를 진행할 수 있도록)
    dummy_categories = ["번역", "감성분석", "코딩", "요약", "설명"]
    all_generated = []
    for cat in dummy_categories:
        for j in range(5):
            all_generated.append({
                "instruction": f"[{cat}] 예시 질문 {j+1}",
                "input": f"입력 예시 {j+1}" if j % 2 == 0 else "",
                "output": f"[{cat}] 답변 예시 {j+1}입니다. 이것은 더미 데이터입니다.",
                "category": cat
            })
    print(f"\n   더미 데이터 {len(all_generated)}개 생성")

In [ ]:
# 생성 데이터 확인
print("📋 생성된 데이터 샘플 확인")
print("=" * 60)

if all_generated:
    # 카테고리별 분포
    from collections import Counter
    cat_counter = Counter(item.get('category', '미분류') for item in all_generated)
    
    print("📊 카테고리별 분포:")
    for cat, count in sorted(cat_counter.items(), key=lambda x: -x[1]):
        bar = "█" * count
        print(f"  {cat:>10s}: {count:>3d} {bar}")
    
    # 랜덤 샘플 표시
    print(f"\n📋 랜덤 샘플 3개:")
    samples = random.sample(all_generated, min(3, len(all_generated)))
    for i, sample in enumerate(samples, 1):
        print(f"\n  {'─'*50}")
        print(f"  📌 샘플 {i} [{sample.get('category', '미분류')}]")
        print(f"  Instruction: {sample['instruction'][:80]}")
        if sample.get('input', '').strip():
            print(f"  Input: {sample['input'][:80]}")
        print(f"  Output: {sample['output'][:100]}...")
else:
    print("⚠️ 생성된 데이터가 없습니다.")

---
## 5️⃣ 6. Distillation: 큰 모델 → 작은 모델 지식 전달

### Distillation의 두 가지 방식

#### 방식 1: 데이터 Distillation (우리가 할 것)
```
GPT-4 (Teacher) → 고품질 응답 생성 → 학습 데이터
                                         ↓
Qwen-1.5B (Student) ← SFT 학습 ← 학습 데이터
```

#### 방식 2: Logit Distillation (전통적 방식)
```
Teacher Model → Soft Labels (확률 분포)
                     ↓
Student Model → 확률 분포를 학습 (KL Divergence)
```

### 데이터 Distillation 전략

| 전략 | 설명 | 효과 |
|------|------|------|
| **Simple Distillation** | GPT-4의 응답을 그대로 학습 | 기본 |
| **Chain-of-Thought** | 추론 과정까지 포함하여 생성 | 추론 능력 향상 |
| **Step-by-Step** | 단계별 설명을 포함 | 설명 능력 향상 |
| **Critique + Revision** | 자가 비평 + 수정 과정 포함 | 자기교정 능력 |

In [ ]:
# Distillation 전략별 프롬프트 설계
print("🔧 Distillation 전략별 프롬프트 설계")
print("=" * 60)

question = "대한민국의 수도는 어디이며, 그 역사적 배경을 설명해주세요."

# 전략 1: Simple Distillation
simple_prompt = f"""다음 질문에 답하세요:
{question}"""

# 전략 2: Chain-of-Thought
cot_prompt = f"""다음 질문에 단계적으로 생각하며 답하세요.
먼저 알고 있는 사실을 나열하고, 그것을 바탕으로 답변을 구성하세요.

질문: {question}

생각 과정:"""

# 전략 3: Step-by-Step
step_prompt = f"""다음 질문에 단계별로 상세하게 설명하세요.
각 단계를 명확히 구분하여 초보자도 이해할 수 있도록 작성하세요.

질문: {question}

단계별 설명:"""

strategies = {
    "Simple": simple_prompt,
    "Chain-of-Thought": cot_prompt,
    "Step-by-Step": step_prompt,
}

for name, prompt in strategies.items():
    print(f"\n{'='*60}")
    print(f"📋 전략: {name}")
    print(f"{'='*60}")
    print(prompt)

print("\n💡 Chain-of-Thought 전략은 Orca 논문에서 효과가 입증되었습니다.")
print("   작은 모델의 추론 능력을 크게 향상시킬 수 있습니다.")

In [ ]:
# Distillation 데이터 생성 함수
print("🔧 Distillation 데이터 생성 함수")
print("=" * 60)

def generate_distillation_data(client, questions, model="gpt-4o-mini",
                                strategy="cot", system_prompt=None):
    """큰 모델로부터 Distillation 데이터 생성"""
    
    if system_prompt is None:
        if strategy == "cot":
            system_prompt = """당신은 전문 교육자입니다. 
질문에 답할 때 먼저 생각 과정을 보여주고, 그 다음 최종 답변을 제공하세요.
형식:
[생각 과정]
...

[답변]
..."""
        elif strategy == "step":
            system_prompt = """당신은 전문 교육자입니다.
질문에 답할 때 단계별로 상세하게 설명하세요.
각 단계는 '1단계:', '2단계:' 형식으로 구분하세요."""
        else:
            system_prompt = "당신은 유능한 AI 어시스턴트입니다. 정확하고 유용한 답변을 제공하세요."
    
    results = []
    total_tokens = 0
    
    if client is None:
        print("⚠️ API 클라이언트 없음 - 더미 결과 반환")
        for q in questions:
            results.append({
                "instruction": q,
                "input": "",
                "output": f"[더미 응답] {q}에 대한 답변입니다.",
                "strategy": strategy
            })
        return results, 0
    
    for i, question in enumerate(questions):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": question}
                ],
                temperature=0.7,
                max_tokens=1000,
            )
            
            answer = response.choices[0].message.content.strip()
            total_tokens += response.usage.total_tokens
            
            results.append({
                "instruction": question,
                "input": "",
                "output": answer,
                "strategy": strategy
            })
            
            print(f"  [{i+1}/{len(questions)}] ✅ 생성 완료 ({len(answer)}자)")
            time.sleep(0.5)
            
        except Exception as e:
            print(f"  [{i+1}/{len(questions)}] ❌ 실패: {e}")
    
    return results, total_tokens

# 테스트 질문
test_questions = [
    "트랜스포머 모델의 어텐션 메커니즘을 설명해주세요.",
    "파이썬에서 데코레이터는 어떻게 동작하나요?",
    "GPT와 BERT의 차이점은 무엇인가요?",
]

print(f"\n📌 테스트 질문 {len(test_questions)}개로 Distillation 데이터 생성...")
distill_data, tokens_used = generate_distillation_data(
    client, test_questions, strategy="cot"
)

if distill_data:
    print(f"\n📋 생성 결과 (첫 번째):")
    print(f"  Q: {distill_data[0]['instruction']}")
    print(f"  A: {distill_data[0]['output'][:200]}...")
    print(f"\n  토큰 사용: {tokens_used:,}")

---
## 6️⃣ 7. 생성 데이터 품질 검증 및 필터링

In [ ]:
# 생성 데이터 품질 검증
print("🔍 생성 데이터 품질 검증 및 필터링")
print("=" * 60)

def quality_filter_generated(data, min_output_len=20, max_output_len=2000):
    """생성된 데이터의 품질 필터링"""
    stats = {
        "total": len(data),
        "passed": 0,
        "removed_short": 0,
        "removed_long": 0,
        "removed_empty": 0,
        "removed_duplicate": 0,
        "removed_low_quality": 0,
    }
    
    filtered = []
    seen_instructions = set()
    
    for item in data:
        instruction = item.get("instruction", "").strip()
        output = item.get("output", "").strip()
        
        # 빈 데이터 체크
        if not instruction or not output:
            stats["removed_empty"] += 1
            continue
        
        # 길이 체크
        if len(output) < min_output_len:
            stats["removed_short"] += 1
            continue
        
        if len(output) > max_output_len:
            stats["removed_long"] += 1
            continue
        
        # 중복 체크
        if instruction in seen_instructions:
            stats["removed_duplicate"] += 1
            continue
        seen_instructions.add(instruction)
        
        # 저품질 체크 (반복 텍스트, 의미없는 응답)
        words = output.split()
        if len(words) > 10:
            unique_ratio = len(set(words)) / len(words)
            if unique_ratio < 0.3:
                stats["removed_low_quality"] += 1
                continue
        
        filtered.append(item)
    
    stats["passed"] = len(filtered)
    
    # 리포트 출력
    print(f"\n📊 품질 필터링 결과:")
    print(f"  입력 데이터: {stats['total']}개")
    print(f"  빈 데이터 제거: {stats['removed_empty']}개")
    print(f"  너무 짧은 응답: {stats['removed_short']}개")
    print(f"  너무 긴 응답: {stats['removed_long']}개")
    print(f"  중복 제거: {stats['removed_duplicate']}개")
    print(f"  저품질 제거: {stats['removed_low_quality']}개")
    print(f"  {'─'*40}")
    print(f"  최종 데이터: {stats['passed']}개 ({stats['passed']/max(stats['total'],1)*100:.1f}%)")
    
    return filtered, stats

# 필터링 실행
if all_generated:
    filtered_data, filter_stats = quality_filter_generated(all_generated)
    
    # 최종 데이터 저장
    final_data = seed_data + filtered_data  # 시드 + 생성 데이터
    
    output_file = os.path.join(DATA_DIR, "synthetic_data.json")
    # category 키 제거하여 저장
    save_data = [
        {k: v for k, v in item.items() if k in ["instruction", "input", "output"]}
        for item in final_data
    ]
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(save_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ 최종 데이터 저장: {output_file}")
    print(f"   시드 데이터: {len(seed_data)}개")
    print(f"   생성 데이터: {len(filtered_data)}개")
    print(f"   합계: {len(final_data)}개")

---
## 7️⃣ 8. 비용 계산 및 최적화 팁

In [ ]:
# API 비용 계산기
print("💰 데이터 생성 비용 계산기")
print("=" * 60)

# 모델별 가격 (2024년 기준, 100만 토큰당)
pricing = {
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4-turbo": {"input": 10.00, "output": 30.00},
    "gpt-3.5-turbo": {"input": 0.50, "output": 1.50},
}

def estimate_cost(n_samples, avg_input_tokens=500, avg_output_tokens=300, model="gpt-4o-mini"):
    """데이터 생성 비용 추정"""
    prices = pricing.get(model, pricing["gpt-4o-mini"])
    
    total_input = n_samples * avg_input_tokens
    total_output = n_samples * avg_output_tokens
    
    input_cost = total_input / 1_000_000 * prices["input"]
    output_cost = total_output / 1_000_000 * prices["output"]
    total_cost = input_cost + output_cost
    
    return {
        "model": model,
        "n_samples": n_samples,
        "total_input_tokens": total_input,
        "total_output_tokens": total_output,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }

# 다양한 시나리오 비용 계산
scenarios = [
    (100, "gpt-4o-mini", "소규모 실습"),
    (1000, "gpt-4o-mini", "중규모 프로젝트"),
    (10000, "gpt-4o-mini", "대규모 프로젝트"),
    (1000, "gpt-4o", "고품질 데이터 (gpt-4o)"),
    (1000, "gpt-4-turbo", "최고 품질 (gpt-4-turbo)"),
]

print(f"\n{'시나리오':>25s} | {'데이터 수':>10s} | {'모델':>15s} | {'예상 비용':>12s}")
print("-" * 75)

for n, model, desc in scenarios:
    result = estimate_cost(n, model=model)
    print(f"{desc:>25s} | {n:>10,d} | {model:>15s} | ${result['total_cost']:>10.2f}")

print("\n💡 비용 최적화 팁:")
print("  1️⃣ gpt-4o-mini 사용 → gpt-4o 대비 약 10~20배 저렴")
print("  2️⃣ 배치 생성: 한 번에 여러 개 생성하면 프롬프트 토큰 절약")
print("  3️⃣ 프롬프트 최적화: 시드 예시 수를 최소화")
print("  4️⃣ 점진적 접근: 100개 → 검증 → 1000개 → 검증 → 확장")
print("  5️⃣ 캐싱: 동일 프롬프트 재사용 방지")

In [ ]:
# 비용 대비 효과 분석
print("📊 데이터 양 vs 성능 (일반적 트렌드)")
print("=" * 60)

print("""
📌 일반적으로 관찰되는 데이터 양 vs 파인튜닝 성능:

  데이터 수  | 예상 품질  | 비용 (4o-mini) | 권장 단계
  ─────────┼──────────┼──────────────┼──────────
    100개   | ★★☆☆☆   |    ~$0.05    | 1. 프로토타이핑
    500개   | ★★★☆☆   |    ~$0.25    | 2. 초기 실험
  1,000개   | ★★★★☆   |    ~$0.50    | 3. 기본 파인튜닝
  5,000개   | ★★★★☆   |    ~$2.50    | 4. 준 프로덕션
 10,000개   | ★★★★★   |    ~$5.00    | 5. 프로덕션
 50,000개   | ★★★★★   |   ~$25.00    | 6. 고성능 (수확체감)

💡 핵심 인사이트:
  - 1,000~5,000개에서 가성비가 가장 좋음
  - 10,000개 이상부터는 수확체감(diminishing returns)
  - 데이터 품질이 양보다 중요 (LIMA 논문)
  - 도메인이 좁을수록 적은 데이터로도 효과적
""")

---
## 📝 9. 정리 및 핵심 요약

### 🎯 이번 세션에서 배운 핵심 개념

| # | 개념 | 핵심 내용 |
|---|------|----------|
| 1️⃣ | **합성 데이터** | LLM이 생성한 인공 학습 데이터 |
| 2️⃣ | **Self-Instruct** | Seed → LLM 생성 → 필터링 → 반복 |
| 3️⃣ | **GPT-4 생성** | 프롬프트 설계 + API 호출 + JSON 파싱 |
| 4️⃣ | **배치 확장** | 10개 시드 → 100개+ 자동 생성 |
| 5️⃣ | **Distillation** | Teacher(GPT-4) → Student(Qwen-1.5B) 지식 전달 |
| 6️⃣ | **품질 필터링** | 길이/중복/품질 자동 검증 |
| 7️⃣ | **비용 최적화** | gpt-4o-mini 사용, 배치 생성, 점진적 확장 |

### 🔑 실무 체크리스트

```
□ 시드 데이터 준비 (카테고리별 최소 3~5개)
□ 생성 프롬프트 설계 및 테스트 (100개 먼저 생성)
□ 품질 검증 (수동 검토 + 자동 필터링)
□ 대규모 생성 (1,000~5,000개)
□ 최종 품질 필터링 및 데이터 포맷 변환
□ 비용 추적 및 최적화
```

### 📚 다음 세션 예고
- **Session 17**: Next Token Prediction 기반 SFT 실습 - 실제 모델 파인튜닝!

In [ ]:
print("\n✅ Session 16 완료!")
print("📚 다음 세션: Next Token Prediction 기반 SFT 실습")
print(f"\n📁 생성된 파일:")
print(f"  1. {os.path.join(DATA_DIR, 'synthetic_data.json')} (합성 데이터)")
print("\n💡 다음 세션에서는 이 데이터를 사용하여 실제 모델을 파인튜닝합니다!")
print("   🔧 준비물: GPU (RTX 4060), Qwen2.5-1.5B-Instruct 모델")